# Tanzania Tourism Prediction — Clean up Version


## 1. Imports & setup


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

RANDOM_STATE = 42
pd.set_option('display.max_columns', 100)


## 2. Load data


In [2]:
train = pd.read_csv('data/Train.csv')
test = pd.read_csv('data/Test.csv')

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')
train.head()


Train: (4809, 23)
Test:  (1601, 22)


,ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,payment_mode,first_trip_tz,most_impressing,total_cost
0,tour_0,SWIZERLAND,45-64,Friends/Relatives,1.0,1.0,Leisure and Holidays,Wildlife tourism,"Friends, relatives",Independent,No,No,No,No,No,No,No,13.0,0.0,Cash,No,Friendly People,674602.5
1,tour_10,UNITED KINGDOM,25-44,NaN,1.0,0.0,Leisure and Holidays,Cultural tourism,others,Independent,No,No,No,No,No,No,No,14.0,7.0,Cash,Yes,"Wonderful Country, Landscape, Nature",3214906.5
2,tour_1000,UNITED KINGDOM,25-44,Alone,0.0,1.0,Visiting Friends and Relatives,Cultural tourism,"Friends, relatives",Independent,No,No,No,No,No,No,No,1.0,31.0,Cash,No,Excellent Experience,3315000.0
3,tour_1002,UNITED KINGDOM,25-44,Spouse,1.0,1.0,Leisure and Holidays,Wildlife tourism,"Travel, agent, tour operator",Package Tour,No,Yes,Yes,Yes,Yes,Yes,No,11.0,0.0,Cash,Yes,Friendly People,7790250.0
4,tour_1004,CHINA,1-24,NaN,1.0,0.0,Leisure and Holidays,Wildlife tourism,"Travel, agent, tour operator",Independent,No,No,No,No,No,No,No,7.0,4.0,Cash,Yes,No comments,1657500.0


## 3. Compact EDA


In [3]:
print('Missing values:')
display(train.isna().sum().sort_values(ascending=False).head(10))

print('\nTarget distribution:')
display(train['total_cost'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(0))

print('\nCategorical cardinality:')
display(train.select_dtypes(include='object').nunique().sort_values(ascending=False))


Missing values:


travel_with             1114
most_impressing          313
total_male                 5
total_female               3
ID                         0
package_transport_tz       0
first_trip_tz              0
payment_mode               0
night_zanzibar             0
night_mainland             0
dtype: int64


Target distribution:


count        4809.0
mean      8114389.0
std      12224903.0
min         49000.0
1%          61000.0
5%         140000.0
25%        812175.0
50%       3397875.0
75%       9945000.0
95%      33481500.0
99%      58317480.0
max      99532875.0
Name: total_cost, dtype: float64


Categorical cardinality:


ID                       4809
country                   105
main_activity               9
info_source                 8
most_impressing             7
purpose                     7
travel_with                 5
age_group                   4
payment_mode                4
tour_arrangement            2
package_transport_int       2
package_food                2
package_transport_tz        2
package_sightseeing         2
package_guided_tour         2
package_insurance           2
first_trip_tz               2
package_accomodation        2
dtype: int64

## 4. Feature engineering

Do not use `total_cost` inside this function.


In [4]:
def add_basic_features(df):
    df = df.copy()
    df = df.drop(columns=['spend_group'], errors='ignore')

    # Clean categorical text
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('object').where(
            df[col].isna(),
            df[col].astype(str).str.strip()
        )

    if 'country' in df.columns:
        df['country'] = df['country'].replace({'SWIZERLAND': 'SWITZERLAND'})

    if 'main_activity' in df.columns:
        df['main_activity'] = df['main_activity'].replace({'business': 'Business'})

    # Package features
    pkg_cols = [
        'package_transport_int', 'package_accomodation', 'package_food',
        'package_transport_tz', 'package_sightseeing', 'package_guided_tour',
        'package_insurance'
    ]

    for col in pkg_cols:
        if col in df.columns:
            df[col] = (df[col].fillna('No') == 'Yes').astype(int)

    used_pkg_cols = [c for c in pkg_cols if c in df.columns]
    df['package_count'] = df[used_pkg_cols].sum(axis=1) if used_pkg_cols else 0

    # Numeric conversion
    for col in ['total_male', 'total_female', 'night_mainland', 'night_zanzibar']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Group size
    if {'total_male', 'total_female'}.issubset(df.columns):
        df['group_size'] = df['total_male'].fillna(0) + df['total_female'].fillna(0)
        df['group_size_log'] = np.log1p(df['group_size'].clip(lower=0))
        df['pct_female'] = np.where(
            df['group_size'] > 0,
            df['total_female'].fillna(0) / df['group_size'],
            0
        )

    # Nights
    if {'night_mainland', 'night_zanzibar'}.issubset(df.columns):
        mainland = df['night_mainland'].clip(lower=0).fillna(0)
        zanzibar = df['night_zanzibar'].clip(lower=0).fillna(0)
        total_nights = mainland + zanzibar

        df['mainland_log'] = np.log1p(mainland)
        df['zanzibar_log'] = np.log1p(zanzibar)
        df['visited_zanzibar'] = (zanzibar > 0).astype(int)
        df['total_nights'] = total_nights
        df['total_nights_log'] = np.log1p(total_nights)
        df['zanzibar_share_nights'] = np.where(total_nights > 0, zanzibar / total_nights, 0)

        if 'group_size' in df.columns:
            df['person_nights'] = total_nights * df['group_size'].fillna(0)
            df['person_nights_log'] = np.log1p(df['person_nights'].clip(lower=0))

    # Missing categorical values
    if 'travel_with' in df.columns:
        df['travel_with'] = df['travel_with'].fillna('Unknown')
        df['is_alone'] = (df['travel_with'] == 'Alone').astype(int)
        df['travel_with_missing'] = (df['travel_with'] == 'Unknown').astype(int)

    if 'most_impressing' in df.columns:
        df['most_impressing'] = df['most_impressing'].fillna('No comments')

    if 'main_activity' in df.columns:
        df['is_wildlife_activity'] = (df['main_activity'] == 'Wildlife tourism').astype(int)

    # Two useful interactions
    if {'purpose', 'travel_with'}.issubset(df.columns):
        df['purpose_x_travel'] = df['purpose'].astype(str) + '_' + df['travel_with'].astype(str)

    if {'country', 'purpose'}.issubset(df.columns):
        df['country_x_purpose'] = df['country'].astype(str) + '_' + df['purpose'].astype(str)

    return df


## 5. Train / validation / submission matrices


In [5]:
train_fe = add_basic_features(train)
test_fe = add_basic_features(test)

X = train_fe.drop(columns=['ID', 'total_cost'], errors='ignore')
y = train_fe['total_cost']

X_submit = test_fe.drop(columns=['ID', 'total_cost'], errors='ignore')
X_submit = X_submit.reindex(columns=X.columns)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_submit:{X_submit.shape}')
print(f'Raw target median: {y.median():,.0f} TZS')
print(f'Raw target max:    {y.max():,.0f} TZS')


X_train: (3847, 38)
X_val:   (962, 38)
X_submit:(1601, 38)
Raw target median: 3,397,875 TZS
Raw target max:    99,532,875 TZS


## 6. Preprocessing with sklearn TargetEncoder

TargetEncoder is used for high-information categorical columns. OneHotEncoder is also kept for categorical columns because it slightly improved validation MAE in this dataset.


In [11]:
target_encode_cols = [
    'country', 'purpose', 'main_activity', 'most_impressing',
    'tour_arrangement', 'age_group', 'info_source', 'payment_mode',
    'travel_with', 'country_x_purpose', 'purpose_x_travel'
]

target_encode_cols = [c for c in target_encode_cols if c in X.columns]
num_features = X.select_dtypes(include='number').columns.tolist()
cat_features = X.select_dtypes(include='object').columns.tolist()


try:
    ohe = OneHotEncoder(handle_unknown='ignore', min_frequency=10, sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', min_frequency=10, sparse=True)

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

onehot_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', ohe)
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('target', TargetEncoder(
        target_type='continuous',
        smooth='auto',
        cv=5,
        random_state=RANDOM_STATE
    ), target_encode_cols),
    ('onehot', onehot_pipeline, cat_features)
])

print(f'Target-encoded columns ({len(target_encode_cols)}):')
print(target_encode_cols)
print(f'Numeric columns ({len(num_features)}):')
print(num_features)
print(f'One-hot columns ({len(cat_features)}):')
print(cat_features)


Target-encoded columns (11):
['country', 'purpose', 'main_activity', 'most_impressing', 'tour_arrangement', 'age_group', 'info_source', 'payment_mode', 'travel_with', 'country_x_purpose', 'purpose_x_travel']
Numeric columns (26):
['total_female', 'total_male', 'package_transport_int', 'package_accomodation', 'package_food', 'package_transport_tz', 'package_sightseeing', 'package_guided_tour', 'package_insurance', 'night_mainland', 'night_zanzibar', 'package_count', 'group_size', 'group_size_log', 'pct_female', 'mainland_log', 'zanzibar_log', 'visited_zanzibar', 'total_nights', 'total_nights_log', 'zanzibar_share_nights', 'person_nights', 'person_nights_log', 'is_alone', 'travel_with_missing', 'is_wildlife_activity']
One-hot columns (12):
['country', 'age_group', 'travel_with', 'purpose', 'main_activity', 'info_source', 'tour_arrangement', 'payment_mode', 'first_trip_tz', 'most_impressing', 'purpose_x_travel', 'country_x_purpose']


## 7. Baseline


In [12]:
baseline = DummyRegressor(strategy='median')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_val)
baseline_mae = mean_absolute_error(y_val, baseline_pred)
print(f'Baseline median MAE: {baseline_mae:,.0f} TZS')


Baseline median MAE: 6,675,876 TZS


## 8. Model 1 — Ridge Regression


In [13]:
pipe_ridge = Pipeline([
    ('preprocessor', preprocessor),
    ('ridge', Ridge(
        alpha=10.0,
        solver='lsqr',
        max_iter=10000,
        random_state=RANDOM_STATE
    ))
])

pipe_ridge.fit(X_train, y_train)
ridge_pred = np.clip(pipe_ridge.predict(X_val), 0, None)
ridge_mae = mean_absolute_error(y_val, ridge_pred)
print(f'Ridge MAE: {ridge_mae:,.0f} TZS')


Ridge MAE: 5,439,976 TZS


## 9. Model 2 — Random Forest


In [14]:
pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

pipe_rf.fit(X_train, y_train)
rf_pred = np.clip(pipe_rf.predict(X_val), 0, None)
rf_mae = mean_absolute_error(y_val, rf_pred)
print(f'Random Forest MAE: {rf_mae:,.0f} TZS')


Random Forest MAE: 4,795,471 TZS


## 10. Model 3 — XGBoost optimized for MAE

The competition metric is MAE, so this model uses raw `total_cost` and `objective='reg:absoluteerror'`. This is usually better here than training on `log1p(total_cost)` and transforming back.


In [15]:
if XGBOOST_AVAILABLE:
    pipe_xgb = Pipeline([
        ('preprocessor', preprocessor),
        ('xgb', XGBRegressor(
            objective='reg:absoluteerror',
            n_estimators=900,
            max_depth=3,
            learning_rate=0.025,
            subsample=0.75,
            colsample_bytree=0.75,
            min_child_weight=3,
            reg_lambda=12,
            reg_alpha=0.3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0
        ))
    ])

    pipe_xgb.fit(X_train, y_train)
    xgb_pred = np.clip(pipe_xgb.predict(X_val), 0, None)
    xgb_mae = mean_absolute_error(y_val, xgb_pred)
    print(f'XGBoost MAE: {xgb_mae:,.0f} TZS')
else:
    xgb_mae = np.nan
    print('XGBoost is not installed.')


XGBoost MAE: 4,423,337 TZS


## 11. Model comparison


In [16]:
results = pd.DataFrame({
    'Model': ['XGBoost', 'Random Forest', 'Ridge Regression', 'Baseline median'],
    'Validation MAE (TZS)': [xgb_mae, rf_mae, ridge_mae, baseline_mae]
}).sort_values('Validation MAE (TZS)')

results.style.format({'Validation MAE (TZS)': '{:,.0f}'})


,Model,Validation MAE (TZS)
0,XGBoost,"4,423,337"
1,Random Forest,"4,795,471"
2,Ridge Regression,"5,439,976"
3,Baseline median,"6,675,876"


## 12. Residual analysis


In [12]:
best_pred = xgb_pred if XGBOOST_AVAILABLE else rf_pred
residuals = y_val - best_pred

diag = pd.DataFrame({
    'actual': y_val,
    'predicted': best_pred,
    'residual': residuals,
    'abs_error': np.abs(residuals)
})

print(f'Mean residual:   {diag["residual"].mean():,.0f}')
print(f'Median residual: {diag["residual"].median():,.0f}')
print(f'Std residual:    {diag["residual"].std():,.0f}')
print(f'MAE:             {diag["abs_error"].mean():,.0f}')

diag['actual_decile'] = pd.qcut(diag['actual'], 10, duplicates='drop')

decile_summary = diag.groupby('actual_decile', observed=True).agg(
    n=('actual', 'size'),
    actual_median=('actual', 'median'),
    predicted_median=('predicted', 'median'),
    median_residual=('residual', 'median'),
    mae=('abs_error', 'mean')
)

decile_summary.style.format('{:,.0f}')


Mean residual:   1,307,374
Median residual: -45,193
Std residual:    8,984,460
MAE:             4,423,337


,n,actual_median,predicted_median,median_residual,mae
actual_decile,,,,,
"(49999.999, 250028.25]",97,"153,500","517,470","-402,537","1,800,584"
"(250028.25, 530400.0]",97,"400,000","745,791","-345,330","1,440,100"
"(530400.0, 994500.0]",105,"730,000","1,159,969","-336,282","1,521,107"
"(994500.0, 2030248.8]",86,"1,589,266","1,741,406","-327,674","2,621,694"
"(2030248.8, 3363812.5]",96,"2,817,750","2,725,683","-13,363","2,879,140"
"(3363812.5, 4997010.4]",96,"4,267,575","4,669,701","-103,297","3,133,805"
"(4997010.4, 7458750.0]",98,"6,006,862","5,597,384","341,775","2,945,270"
"(7458750.0, 11754990.0]",94,"9,167,034","8,572,509","924,378","3,862,846"
"(11754990.0, 20486700.0]",96,"15,423,038","12,010,940","3,600,892","4,832,587"


## 13. Blending: simple two-XGBoost blend

It adds one extra model, but often improves MAE because the raw-MAE model and log-target model make different errors.


In [13]:
USE_BLEND = True

if XGBOOST_AVAILABLE and USE_BLEND:
    pipe_xgb_log = Pipeline([
        ('preprocessor', preprocessor),
        ('xgb', XGBRegressor(
            objective='reg:squarederror',
            n_estimators=900,
            max_depth=3,
            learning_rate=0.025,
            subsample=0.75,
            colsample_bytree=0.75,
            min_child_weight=3,
            reg_lambda=12,
            reg_alpha=0.3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0
        ))
    ])

    pipe_xgb_log.fit(X_train, np.log1p(y_train))
    xgb_log_pred = np.clip(np.expm1(pipe_xgb_log.predict(X_val)), 0, None)

    blend_pred = 0.7 * xgb_pred + 0.3 * xgb_log_pred
    blend_mae = mean_absolute_error(y_val, blend_pred)

    print(f'XGBoost raw MAE:   {xgb_mae:,.0f} TZS')
    print(f'XGBoost log MAE:   {mean_absolute_error(y_val, xgb_log_pred):,.0f} TZS')
    print(f'Blended MAE:       {blend_mae:,.0f} TZS')
else:
    blend_mae = np.nan


XGBoost raw MAE:   4,423,337 TZS
XGBoost log MAE:   4,529,094 TZS
Blended MAE:       4,413,167 TZS


In [15]:
best_pred = blend_pred
residuals = y_val - best_pred

diag = pd.DataFrame({
    'actual': y_val,
    'predicted': best_pred,
    'residual': residuals,
    'abs_error': np.abs(residuals)
})

print(f'Mean residual:   {diag["residual"].mean():,.0f}')
print(f'Median residual: {diag["residual"].median():,.0f}')
print(f'Std residual:    {diag["residual"].std():,.0f}')
print(f'MAE:             {diag["abs_error"].mean():,.0f}')

diag['actual_decile'] = pd.qcut(diag['actual'], 10, duplicates='drop')

decile_summary = diag.groupby('actual_decile', observed=True).agg(
    n=('actual', 'size'),
    actual_median=('actual', 'median'),
    predicted_median=('predicted', 'median'),
    median_residual=('residual', 'median'),
    mae=('abs_error', 'mean')
)

decile_summary.style.format('{:,.0f}')


Mean residual:   1,605,135
Median residual: 17,894
Std residual:    8,972,523
MAE:             4,413,167


,n,actual_median,predicted_median,median_residual,mae
actual_decile,,,,,
"(49999.999, 250028.25]",97,"153,500","546,993","-361,992","1,701,829"
"(250028.25, 530400.0]",97,"400,000","739,502","-334,168","1,326,343"
"(530400.0, 994500.0]",105,"730,000","1,045,242","-292,338","1,424,157"
"(994500.0, 2030248.8]",86,"1,589,266","1,593,736","-250,665","2,403,743"
"(2030248.8, 3363812.5]",96,"2,817,750","2,543,454","255,766","2,743,697"
"(3363812.5, 4997010.4]",96,"4,267,575","4,493,893",-351,"3,050,510"
"(4997010.4, 7458750.0]",98,"6,006,862","5,319,035","569,870","2,786,229"
"(7458750.0, 11754990.0]",94,"9,167,034","8,082,866","1,081,916","3,620,224"
"(11754990.0, 20486700.0]",96,"15,423,038","10,918,190","4,381,823","5,202,584"
